<a href="https://colab.research.google.com/github/anmollate/Data-Science/blob/main/OneHotEncoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OneHotEncoder is used to encode the nominal categorical columns in input data

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df=pd.read_csv("/content/drive/MyDrive/car_dataset_500_rows.csv")

In [ ]:
df.head(5)

,brand,km_driven,owner,fuel,selling_price
0,Toyota,97488,fourth or higher hand,electric,1755693
1,Honda,129198,test drive car,diesel,144325
2,Honda,142288,second hand,diesel,933731
3,Audi,66582,fourth or higher hand,cng,466066
4,Audi,44570,first hand,cng,756945


In [ ]:
df["fuel"].unique()

array(['electric', 'diesel', 'cng', 'petrol'], dtype=object)

In [ ]:
df["owner"].unique()

array(['fourth or higher hand', 'test drive car', 'second hand',
       'first hand', 'third hand'], dtype=object)

#using pandas to hot encode the columns "owner" and "fuel"

In [ ]:
#using pandas to hot encode the columns "owner" and "fuel"
df_encoded=pd.get_dummies(df,columns=["owner","fuel"],dtype="int") #without specifying the dtype it would encode in boolean values

In [ ]:
df_encoded.head(5)

,brand,km_driven,selling_price,owner_first hand,owner_fourth or higher hand,owner_second hand,owner_test drive car,owner_third hand,fuel_cng,fuel_diesel,fuel_electric,fuel_petrol
0,Toyota,97488,1755693,0,1,0,0,0,0,0,1,0
1,Honda,129198,144325,0,0,0,1,0,0,1,0,0
2,Honda,142288,933731,0,0,1,0,0,0,1,0,0
3,Audi,66582,466066,0,1,0,0,0,1,0,0,0
4,Audi,44570,756945,1,0,0,0,0,1,0,0,0


In [ ]:
df_encoded.shape #it would be 500,12 500rows and extended cols 12

(500, 12)

In [ ]:
#now to avoid the dummy variable trap
df_encoded2=pd.get_dummies(df,columns=["owner","fuel"],drop_first=True,dtype="int")

In [ ]:
df_encoded2.head(5)

,brand,km_driven,selling_price,owner_fourth or higher hand,owner_second hand,owner_test drive car,owner_third hand,fuel_diesel,fuel_electric,fuel_petrol
0,Toyota,97488,1755693,1,0,0,0,0,1,0
1,Honda,129198,144325,0,0,1,0,1,0,0
2,Honda,142288,933731,0,1,0,0,1,0,0
3,Audi,66582,466066,1,0,0,0,0,0,0
4,Audi,44570,756945,0,0,0,0,0,0,0


In [ ]:
df_encoded2.shape #it would be 500 rows and 10 cols as one column from each row is eliminated

(500, 10)

# Now using sklearn to do the same thing

In [ ]:
df.head(5)

,brand,km_driven,owner,fuel,selling_price
0,Toyota,97488,fourth or higher hand,electric,1755693
1,Honda,129198,test drive car,diesel,144325
2,Honda,142288,second hand,diesel,933731
3,Audi,66582,fourth or higher hand,cng,466066
4,Audi,44570,first hand,cng,756945


In [ ]:
x=df[["brand","km_driven","owner","fuel"]]
y=df["selling_price"]

In [ ]:
x.head(5)

,brand,km_driven,owner,fuel
0,Toyota,97488,fourth or higher hand,electric
1,Honda,129198,test drive car,diesel
2,Honda,142288,second hand,diesel
3,Audi,66582,fourth or higher hand,cng
4,Audi,44570,first hand,cng


In [ ]:
y.head(5)

,selling_price
0,1755693
1,144325
2,933731
3,466066
4,756945


In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.3,random_state=42)

In [ ]:
x_train.head(5)

,brand,km_driven,owner,fuel
5,Mercedes,13060,first hand,diesel
116,BMW,137896,first hand,cng
45,BMW,10082,third hand,diesel
16,Hyundai,93931,test drive car,electric
462,BMW,84100,second hand,petrol


In [ ]:
x_test.head(5)

,brand,km_driven,owner,fuel
361,Toyota,22422,second hand,petrol
73,Toyota,113790,second hand,cng
374,Toyota,101333,fourth or higher hand,cng
155,Toyota,71711,second hand,electric
104,Honda,43095,third hand,petrol


In [ ]:
#Now we have to OneHotEncode the columns "fuel" and "owner" the output of the encoding would be in array that would be converted into dataframe and merged remaining cols
from sklearn.preprocessing import OneHotEncoder
encoder=OneHotEncoder(drop="first",sparse_output=False,dtype="int32")

In [ ]:
encoder.fit(x_train[["owner","fuel"]])
encoded_labels=encoder.get_feature_names_out(["owner","fuel"])
x_train_encoded=encoder.transform(x_train[["owner","fuel"]])
x_test_encoded=encoder.transform(x_test[["owner","fuel"]])

In [ ]:
encoded_labels

array(['owner_fourth or higher hand', 'owner_second hand',
       'owner_test drive car', 'owner_third hand', 'fuel_diesel',
       'fuel_electric', 'fuel_petrol'], dtype=object)

In [ ]:
x_train_encoded.shape

(350, 7)

In [ ]:
x_test_encoded.shape

(150, 7)

In [ ]:
x_train.head(5)

,brand,km_driven,owner,fuel
5,Mercedes,13060,first hand,diesel
116,BMW,137896,first hand,cng
45,BMW,10082,third hand,diesel
16,Hyundai,93931,test drive car,electric
462,BMW,84100,second hand,petrol


In [ ]:
final_array=np.hstack((x_train[["brand","km_driven"]].values,x_train_encoded))
final_array

array([['Mercedes', 13060, 0, ..., 1, 0, 0],
       ['BMW', 137896, 0, ..., 0, 0, 0],
       ['BMW', 10082, 0, ..., 1, 0, 0],
       ...,
       ['Hyundai', 72055, 0, ..., 0, 0, 1],
       ['Audi', 131007, 0, ..., 1, 0, 0],
       ['Honda', 133641, 0, ..., 0, 0, 0]], dtype=object)

In [ ]:
final_df=pd.DataFrame(final_array,columns=["brand","km_driven"]+list(encoded_labels))

In [ ]:
final_df.head(5)

,brand,km_driven,owner_fourth or higher hand,owner_second hand,owner_test drive car,owner_third hand,fuel_diesel,fuel_electric,fuel_petrol
0,Mercedes,13060,0,0,0,0,1,0,0
1,BMW,137896,0,0,0,0,0,0,0
2,BMW,10082,0,0,0,1,1,0,0
3,Hyundai,93931,0,0,1,0,0,1,0
4,BMW,84100,0,1,0,0,0,0,1
